# Kubernetes 

## Table of Contents

1. [Kubernetes Fundamentals](#k8s-fundamentals)
2. [Kubernetes Architecture](#k8s-architecture)
3. [Kubernetes Objects](#k8s-objects)
4. [Kubernetes Networking](#k8s-networking)
5.  [Storage in Kubernetes](#k8s-storage)
6.  [Autoscaling & Resource Management](#autoscaling)
7.  [Production Best Practices](#best-practices)

---


## Kubernetes Fundamentals 

### What Problem Does Kubernetes Solve?

You've containerized your application with Docker. Great! But now you face new challenges:

**1. Orchestration:**
You have 50 containers across 10 servers. How do you:
- Decide which server runs which container?
- Move containers if a server fails?
- Balance load across containers?

**2. Scaling:**
- Traffic spikes to 10x normal. How do you automatically add more containers?
- Traffic drops. How do you scale down to save costs?

**3. Updates:**
- You have a new version. How do you update 50 containers without downtime?
- New version has bugs. How do you rollback?

**4. Service Discovery:**
- Containers have dynamic IP addresses. How does one find another?
- Container crashes and restarts with new IP. How do others find it?

**5. Health Management:**
- Container crashes. How do you detect and restart it?
- Container is running but application is hung. How do you detect and fix?

**6. Configuration:**
- 50 containers need the same database URL. How do you update it everywhere?
- Different environments (dev/staging/prod) need different configs. How do you manage?

**Kubernetes answers all these questions.**

### What Kubernetes Provides

**Automated Scheduling:**
You tell Kubernetes: "Run 10 copies of this container, each needs 2GB RAM."
Kubernetes finds servers with available resources and places containers optimally.

**Self-Healing:**
Container crashes? Kubernetes automatically restarts it.
Server dies? Kubernetes moves containers to healthy servers.
Health check fails? Kubernetes replaces container.

**Horizontal Scaling:**
```bash
kubectl scale deployment myapp --replicas=20
```
Instantly scales from 10 to 20 containers.

Or enable autoscaling:
```yaml
# Automatically scale between 2-10 based on CPU usage
HorizontalPodAutoscaler: min=2, max=10, targetCPU=70%
```

**Service Discovery & Load Balancing:**
Kubernetes gives your application a stable DNS name.
Containers come and go, but the name remains constant.
Built-in load balancing across all containers.

**Automated Rollouts & Rollbacks:**
```bash
kubectl set image deployment/myapp myapp=myapp:v2
```
Kubernetes gradually replaces old containers with new ones.
Zero downtime. If v2 has issues:
```bash
kubectl rollout undo deployment/myapp
```
Instant rollback to v1.

**Secret & Configuration Management:**
Store configurations separately from code.
Update configs without rebuilding containers.
Secure storage for sensitive data (passwords, API keys).

**Storage Orchestration:**
Automatically attach storage to containers.
Storage follows container if it moves to different server.

### Kubernetes Architecture - How It Works

Kubernetes cluster = Control Plane + Worker Nodes

**Control Plane (Brain):**
Makes global decisions about cluster.
Manages cluster state.
Responds to cluster events.

**Worker Nodes (Muscle):**
Run your containerized applications.
Managed by control plane.

```
┌─────────────────── Kubernetes Cluster ───────────────────┐
│                                                           │
│  ┌────────── Control Plane ──────────┐                   │
│  │                                   │                   │
│  │  ┌──────────────┐                │                   │
│  │  │  API Server  │◄───────────────┼───── kubectl      │
│  │  └──────┬───────┘                │                   │
│  │         │                        │                   │
│  │  ┌──────▼────┐  ┌──────────────┐│                   │
│  │  │   etcd    │  │  Scheduler   ││                   │
│  │  │ (Database)│  │(Pod Placement││                   │
│  │  └───────────┘  └──────────────┘│                   │
│  │                                  │                   │
│  │  ┌─────────────────────────────┐│                   │
│  │  │   Controller Manager        ││                   │
│  │  │ (Maintains Desired State)   ││                   │
│  │  └─────────────────────────────┘│                   │
│  └──────────────────────────────────┘                   │
│                                                          │
│  ┌──── Worker Node 1 ────┐  ┌──── Worker Node 2 ────┐  │
│  │                        │  │                        │  │
│  │  ┌────────────────┐   │  │  ┌────────────────┐   │  │
│  │  │    kubelet     │   │  │  │    kubelet     │   │  │
│  │  │(Node Agent)    │   │  │  │(Node Agent)    │   │  │
│  │  └────────────────┘   │  │  └────────────────┘   │  │
│  │                        │  │                        │  │
│  │  ┌────────────────┐   │  │  ┌────────────────┐   │  │
│  │  │  kube-proxy    │   │  │  │  kube-proxy    │   │  │
│  │  │(Network Proxy) │   │  │  │(Network Proxy) │   │  │
│  │  └────────────────┘   │  │  └────────────────┘   │  │
│  │                        │  │                        │  │
│  │  ┌─────┐  ┌─────┐    │  │  ┌─────┐  ┌─────┐    │  │
│  │  │Pod 1│  │Pod 2│    │  │  │Pod 3│  │Pod 4│    │  │
│  │  └─────┘  └─────┘    │  │  └─────┘  └─────┘    │  │
│  └────────────────────────┘  └────────────────────────┘  │
└───────────────────────────────────────────────────────────┘
```

### Component Deep Dive

**API Server:**
The frontend of Kubernetes. Everything goes through API server.

```
You: kubectl apply -f deployment.yaml
     ↓
API Server:
  1. Authenticates you (who are you?)
  2. Authorizes you (can you do this?)
  3. Validates request (is YAML correct?)
  4. Saves to etcd (store desired state)
  5. Notifies relevant controllers
```

**etcd:**
Distributed key-value database. Stores entire cluster state.

```
etcd stores:
- What pods should be running?
- On which nodes?
- What's the current status?
- All configuration data
- Secrets
- Service definitions
- Everything!
```

If etcd is lost, entire cluster state is lost. Always backup etcd!

**Scheduler:**
Decides which worker node should run each pod.

```
New pod created (status: Pending)
     ↓
Scheduler watches for pending pods
     ↓
Scheduler evaluates all nodes:
  - Enough CPU? ✓
  - Enough memory? ✓
  - Satisfies constraints? ✓
  - Best fit? Node 3!
     ↓
Scheduler assigns pod to Node 3
     ↓
Updates API server: pod → Node 3
```

Factors scheduler considers:
- Resource requirements (CPU, memory)
- Hardware constraints (needs SSD? GPU?)
- Affinity rules (run near database pods?)
- Anti-affinity rules (spread across zones?)
- Taints and tolerations (nodes with special purposes?)

**Controller Manager:**
Runs multiple controllers. Each controller watches for changes and takes action to maintain desired state.

**Node Controller:** Monitors node health
```
Node hasn't reported in 40 seconds
  ↓
Mark node as Unknown
  ↓
After 5 minutes, evict all pods from node
  ↓
Schedule pods on healthy nodes
```

**Replication Controller:** Maintains pod count
```
Deployment specifies: 3 replicas
Currently running: 2 replicas
  ↓
Create 1 new pod
```

**Endpoints Controller:** Populates endpoint objects
```
Service selector: app=myapp
  ↓
Find all pods with label app=myapp
  ↓
Create endpoints with their IP addresses
  ↓
Service routes traffic to these IPs
```

**kubelet (on each worker node):**
Node agent. Ensures containers are running in pods.

```
API Server: "Run pod X on this node"
  ↓
kubelet:
  1. Pull container image
  2. Start container
  3. Monitor container health
  4. Report status to API server
  5. Restart container if it fails
```

kubelet doesn't manage containers Docker created outside Kubernetes.

**kube-proxy (on each worker node):**
Network proxy. Implements Kubernetes Service abstraction.

```
Service "api-service" created (ClusterIP: 10.96.0.10)
Backend pods: 10.1.1.2, 10.1.1.3, 10.1.1.4
  ↓
kube-proxy sets up iptables rules:
  Traffic to 10.96.0.10 → randomly route to one of:
    - 10.1.1.2
    - 10.1.1.3
    - 10.1.1.4
```

### How Components Work Together - Example

You run: `kubectl apply -f deployment.yaml`

```
1. kubectl → API Server (HTTPS request)

2. API Server:
   - Authenticates you
   - Authorizes action
   - Validates YAML
   - Writes to etcd

3. Deployment Controller (in Controller Manager):
   - Watches API server for Deployment changes
   - Sees new Deployment
   - Creates ReplicaSet object
   - Writes to API server

4. ReplicaSet Controller:
   - Watches for ReplicaSet changes
   - Sees new ReplicaSet (replicas: 3)
   - Creates 3 Pod objects
   - Writes to API server

5. Scheduler:
   - Watches for unscheduled Pods
   - Sees 3 new Pods (status: Pending)
   - Evaluates nodes, picks best fit
   - Assigns each Pod to a node
   - Updates API server

6. kubelet (on each assigned node):
   - Watches API server for Pods assigned to it
   - Sees new Pod assignment
   - Pulls container image
   - Starts container
   - Reports status to API server

7. kube-proxy (on each node):
   - Watches for Service changes (if Service created)
   - Sets up iptables rules
   - Enables networking to Pods

Throughout:
- etcd stores every state change
- Controllers continuously reconcile desired vs actual state
- kubelet continuously monitors container health
```

This happens in seconds, automatically, across potentially hundreds of machines!

---



## Kubernetes Objects

### Pod - The Fundamental Unit

**What is a Pod?**

A Pod is the smallest deployable unit in Kubernetes. It's a wrapper around one or more containers.

**Why Pods, Not Just Containers?**

Sometimes multiple containers need to work together as a single unit:
- Main application container + logging sidecar
- Main container + monitoring agent
- Main container + proxy

These containers need to:
- Share the same network (reach each other via localhost)
- Share storage volumes
- Be scheduled together on the same node
- Scale together

Pod provides this grouping.

**Pod Anatomy:**

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: myapp-pod
  labels:
    app: myapp
    version: v1
    environment: production
  # Labels are key-value pairs
  # Used for selecting, grouping, querying pods
  
spec:
  containers:
  - name: myapp-container
    image: myapp:v1
    # Image pulled from registry
    
    ports:
    - containerPort: 8000
      # Container listens on 8000
      # This is documentation only!
      # Container can listen on any port regardless of this
    
    env:
    - name: DATABASE_URL
      value: "postgresql://db:5432/mydb"
    # Environment variables
    
    resources:
      requests:
        memory: "256Mi"
        cpu: "250m"
      # Minimum guaranteed resources
      # Scheduler uses this to place pod
      
      limits:
        memory: "512Mi"
        cpu: "500m"
      # Maximum allowed resources
      # If exceeded, container is throttled (CPU) or killed (memory)
    
    volumeMounts:
    - name: data
      mountPath: /app/data
      # Mount volume named 'data' at /app/data
  
  volumes:
  - name: data
    emptyDir: {}
    # emptyDir: temporary directory, deleted when pod deleted
  
  restartPolicy: Always
  # Always: restart container if it exits (default)
  # OnFailure: only if exit code != 0
  # Never: never restart
```

**Pod Lifecycle:**

```
Pending → Running → Succeeded/Failed

Pending:
  - Pod created but containers not yet started
  - Reasons: image pulling, scheduling, waiting for resources

Running:
  - At least one container running
  - Or container restarting

Succeeded:
  - All containers exited with status 0
  - Won't be restarted (unless restartPolicy)

Failed:
  - All containers exited, at least one with non-zero status
```

**Multi-Container Pod Example (Sidecar Pattern):**

```yaml
spec:
  containers:
  # Main application
  - name: app
    image: myapp:v1
    volumeMounts:
    - name: logs
      mountPath: /var/log/app
  
  # Logging sidecar
  - name: log-shipper
    image: fluentd:latest
    volumeMounts:
    - name: logs
      mountPath: /var/log/app
    # Reads logs from shared volume, ships to central logging
  
  volumes:
  - name: logs
    emptyDir: {}
```

Both containers share the `logs` volume. App writes logs, fluentd reads and ships them.

**Why You Don't Create Pods Directly:**

Pods are mortal. When a pod dies, it's gone forever (not restarted unless you manually recreate it).

Instead, use higher-level objects (Deployments, StatefulSets) that manage pods for you.

### ReplicaSet - Maintaining Pod Count

**Purpose:** Ensure a specified number of pod replicas are running at all times.

```yaml
apiVersion: apps/v1
kind: ReplicaSet
metadata:
  name: myapp-replicaset
spec:
  replicas: 3
  # Maintain exactly 3 pods
  
  selector:
    matchLabels:
      app: myapp
  # ReplicaSet manages pods with this label
  
  template:
    # Pod template - blueprint for creating pods
    metadata:
      labels:
        app: myapp
        # Must match selector above!
    spec:
      containers:
      - name: myapp
        image: myapp:v1
```

**How It Works:**

```
Desired state: 3 replicas
Current state: 2 replicas
  ↓
ReplicaSet Controller: Create 1 more pod
  ↓
Creates pod from template
  ↓
Current state: 3 replicas ✓
```

If a pod crashes:
```
Desired: 3
Current: 2 (one pod died)
  ↓
ReplicaSet: Create replacement pod
  ↓
Current: 3 ✓
```

**Why You Don't Use ReplicaSets Directly:**

ReplicaSets are low-level. They don't provide:
- Rolling updates
- Rollback capability
- Update strategies

Use Deployments instead (which create ReplicaSets automatically).

### Deployment - Declarative Updates

**Most Common Object:** 99% of the time, you'll use Deployments to run stateless applications.

**What Deployment Provides:**
- Declarative updates (describe what you want, K8s figures out how)
- Rolling updates with zero downtime
- Rollback capability
- Version history
- Pause/resume updates

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: myapp-deployment
  labels:
    app: myapp
spec:
  replicas: 3
  
  strategy:
    type: RollingUpdate
    # Two strategies: RollingUpdate (default) or Recreate
    
    rollingUpdate:
      maxSurge: 1
      # Max number of pods above desired count during update
      # With replicas=3, maxSurge=1: can have 4 pods temporarily
      
      maxUnavailable: 1
      # Max number of pods unavailable during update
      # With replicas=3, maxUnavailable=1: at least 2 must be running
  
  selector:
    matchLabels:
      app: myapp
  
  template:
    metadata:
      labels:
        app: myapp
        version: v1
    
    spec:
      containers:
      - name: myapp
        image: myapp:v1
        ports:
        - containerPort: 8000
        
        # Health Checks (Critical!)
        livenessProbe:
          # Is container alive?
          # If fails, kubelet kills container
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 30
          # Wait 30s after container starts before first check
          # Gives app time to initialize
          periodSeconds: 10
          # Check every 10 seconds
          timeoutSeconds: 5
          # If no response in 5s, consider failed
          failureThreshold: 3
          # Fail 3 times before killing container
        
        readinessProbe:
          # Is container ready to serve traffic?
          # If fails, removed from service endpoints
          httpGet:
            path: /ready
            port: 8000
          initialDelaySeconds: 5
          periodSeconds: 5
          
        # Difference between liveness and readiness:
        # Liveness: Should container be restarted? (App crashed/hung)
        # Readiness: Should container receive traffic? (App still loading data)
        
        resources:
          requests:
            memory: "256Mi"
            cpu: "250m"
          limits:
            memory: "512Mi"
            cpu: "500m"
```

**Rolling Update Process:**

You update image: `myapp:v1` → `myapp:v2`

```
Initial state: 3 pods running v1

Step 1: Create 1 pod with v2 (maxSurge=1)
  [v1] [v1] [v1] [v2]
  Running: 4 pods (3 old + 1 new)

Step 2: Wait for v2 pod to be ready (readinessProbe passes)
  [v1] [v1] [v1] [v2✓]

Step 3: Terminate 1 v1 pod (maxUnavailable=1)
  [v1] [v1] [v2✓]
  Running: 3 pods (2 old + 1 new)

Step 4: Create another v2 pod
  [v1] [v1] [v2✓] [v2]
  Running: 4 pods (2 old + 2 new)

Step 5: Wait for v2 pod ready
  [v1] [v1] [v2✓] [v2✓]

Step 6: Terminate another v1 pod
  [v1] [v2✓] [v2✓]

Step 7: Create final v2 pod
  [v1] [v2✓] [v2✓] [v2]

Step 8: Wait for ready, terminate last v1
  [v2✓] [v2✓] [v2✓]

Final state: 3 pods running v2
```

At any point during update:
- At least 2 pods serving traffic (3 - maxUnavailable)
- At most 4 pods total (3 + maxSurge)
- Zero downtime!

**Rollback:**

New version has bugs:

```bash
kubectl rollout undo deployment/myapp-deployment
```

Kubernetes instantly reverses the process, rolling back to v1.

**Deployment Commands:**

```bash
# Create deployment
kubectl apply -f deployment.yaml

# Get deployments
kubectl get deployments
# Shows: NAME, READY, UP-TO-DATE, AVAILABLE, AGE

# Describe deployment (detailed info)
kubectl describe deployment myapp-deployment

# Update image
kubectl set image deployment/myapp-deployment myapp=myapp:v2

# Scale deployment
kubectl scale deployment myapp-deployment --replicas=5

# Check rollout status
kubectl rollout status deployment/myapp-deployment
# Shows progress of rolling update

# Pause rollout (stop update mid-way)
kubectl rollout pause deployment/myapp-deployment
# Useful for canary testing

# Resume rollout
kubectl rollout resume deployment/myapp-deployment

# Rollout history
kubectl rollout history deployment/myapp-deployment
# Shows revision history

# Rollback to previous version
kubectl rollout undo deployment/myapp-deployment

# Rollback to specific revision
kubectl rollout undo deployment/myapp-deployment --to-revision=2

# Edit deployment (opens in editor)
kubectl edit deployment myapp-deployment
```

**Recreate Strategy:**

Alternative to RollingUpdate:

```yaml
strategy:
  type: Recreate
```

Process:
1. Kill all old pods
2. Wait for termination
3. Create all new pods

Result: Downtime during update, but simpler and faster. Use for development or when app doesn't support running multiple versions simultaneously.

### Service - Stable Network Endpoint

**The Problem:**

Pods are ephemeral. They get IPs when created, but:
- Pod dies, new pod has different IP
- Pods scale up/down, IPs change
- How do other pods find them?

**The Solution: Service**

Service provides:
- Stable DNS name
- Stable IP address (ClusterIP)
- Load balancing across pods
- Service discovery

```yaml
apiVersion: v1
kind: Service
metadata:
  name: myapp-service
spec:
  type: ClusterIP
  # Service types: ClusterIP, NodePort, LoadBalancer, ExternalName
  
  selector:
    app: myapp
  # Service routes traffic to pods with label app=myapp
  
  ports:
  - protocol: TCP
    port: 80
    # Service listens on port 80
    targetPort: 8000
    # Forwards to pod's port 8000
```

**How It Works:**

```
1. You create Service with selector app=myapp

2. Endpoints Controller finds all pods with label app=myapp
   Suppose it finds:
   - Pod A: 10.1.1.2:8000
   - Pod B: 10.1.1.3:8000
   - Pod C: 10.1.1.4:8000

3. Controller creates Endpoints object:
   myapp-service → [10.1.1.2:8000, 10.1.1.3:8000, 10.1.1.4:8000]

4. Service gets ClusterIP: 10.96.0.10

5. kube-proxy on each node sets up iptables rules:
   Traffic to 10.96.0.10:80 → randomly route to one of:
     - 10.1.1.2:8000
     - 10.1.1.3:8000
     - 10.1.1.4:8000

6. DNS entry created: myapp-service.default.svc.cluster.local → 10.96.0.10
```

**Accessing Service:**

From any pod in the cluster:

```python
# Via ClusterIP
requests.get('http://10.96.0.10')

# Via DNS (short form, same namespace)
requests.get('http://myapp-service')

# Via DNS (full form, any namespace)
requests.get('http://myapp-service.default.svc.cluster.local')
```

**Service Types Explained:**

**1. ClusterIP (Default):**

Internal-only. Only accessible from within cluster.

```yaml
type: ClusterIP
```

Use for: Backend services, databases, internal APIs.

**2. NodePort:**

Exposes service on each node's IP at a static port.

```yaml
type: NodePort
ports:
- port: 80
  targetPort: 8000
  nodePort: 30080
  # Range: 30000-32767
```

Now accessible at: `<any-node-ip>:30080`

```
User → Node1:30080 → Service → Pod
User → Node2:30080 → Service → Pod
User → Node3:30080 → Service → Pod
```

All nodes forward port 30080 to service, even if pod not on that node.

Use for: Development, testing, or when you don't have LoadBalancer.

**3. LoadBalancer:**

Provisions cloud load balancer (AWS ELB, GCP Load Balancer, Azure Load Balancer).

```yaml
type: LoadBalancer
```

Cloud provider creates external load balancer with public IP.

```
Internet → Cloud Load Balancer (1.2.3.4) → NodePort → Service → Pods
```

Use for: Production services that need external access.

**4. ExternalName:**

Maps service to DNS name (CNAME record).

```yaml
type: ExternalName
externalName: api.example.com
```

Pods accessing `myservice` actually connect to `api.example.com`.

Use for: Accessing external services through K8s service abstraction.

**Headless Service:**

Sometimes you don't want load balancing. You want to connect to specific pods.

```yaml
spec:
  clusterIP: None
  # Headless service
```

DNS returns all pod IPs instead of single service IP.

```
myapp-service.default.svc.cluster.local →
  10.1.1.2
  10.1.1.3
  10.1.1.4
```

Use for: Databases (need to connect to specific primary), distributed systems.

### Ingress - HTTP Routing

**Problem with Services:**

Each service needs its own LoadBalancer (expensive!) or NodePort (ugly ports).

```
myapp.com → LoadBalancer 1 → api-service
shop.com → LoadBalancer 2 → shop-service
blog.com → LoadBalancer 3 → blog-service
```

Three load balancers, three IPs, high cost.

**Ingress Solution:**

Single load balancer, intelligent HTTP routing.

```
myapp.com/api → Ingress Controller → api-service
myapp.com/shop → Ingress Controller → shop-service
myapp.com/blog → Ingress Controller → blog-service
```

One load balancer, one IP, route based on hostname and path.

**Ingress Resource:**

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: myapp-ingress
  annotations:
    # Annotations configure ingress controller behavior
    nginx.ingress.kubernetes.io/rewrite-target: /
    # Rewrite /api/foo to /foo before forwarding
    cert-manager.io/cluster-issuer: "letsencrypt-prod"
    # Automatically provision SSL certificate
spec:
  ingressClassName: nginx
  # Which ingress controller to use
  
  tls:
  - hosts:
    - myapp.example.com
    secretName: myapp-tls
    # SSL certificate stored in this secret
  
  rules:
  - host: myapp.example.com
    # Routing rules for this hostname
    http:
      paths:
      - path: /api
        pathType: Prefix
        # /api, /api/users, /api/anything matches
        backend:
          service:
            name: api-service
            port:
              number: 80
      
      - path: /web
        pathType: Prefix
        backend:
          service:
            name: web-service
            port:
              number: 80
      
      - path: /
        pathType: Prefix
        backend:
          service:
            name: default-service
            port:
              number: 80
  
  - host: shop.example.com
    # Different hostname, different backend
    http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: shop-service
            port:
              number: 80
```

**How It Works:**

```
1. User: https://myapp.example.com/api/users

2. DNS resolves to Ingress Controller's load balancer IP

3. HTTPS to Ingress Controller

4. Ingress Controller:
   - Terminates SSL
   - Checks Ingress rules
   - Matches: host=myapp.example.com, path=/api
   - Routes to api-service:80

5. api-service routes to api pod

6. Response: api pod → api-service → Ingress Controller → User
```

**PathType Explained:**

**Prefix:** Match path prefix

```yaml
path: /api
pathType: Prefix
```

Matches: `/api`, `/api/`, `/api/users`, `/api/users/123`

**Exact:** Exact match only

```yaml
path: /api
pathType: Exact
```

Matches: `/api`, `/api/`
Doesn't match: `/api/users`

**ImplementationSpecific:** Depends on ingress controller

**Ingress Controllers:**

Ingress resource is just configuration. Need Ingress Controller to actually route traffic.

Popular controllers:
- **NGINX Ingress:** Most popular, feature-rich
- **Traefik:** Simple, good for small deployments
- **HAProxy:** High performance
- **AWS ALB Ingress:** Native AWS integration
- **Kong:** API gateway features

**Install NGINX Ingress Controller:**

```bash
kubectl apply -f https://raw.githubusercontent.com/kubernetes/ingress-nginx/controller-v1.8.1/deploy/static/provider/cloud/deploy.yaml

# This creates:
# - Deployment (ingress controller pods)
# - Service (type: LoadBalancer)
# - ConfigMaps, ServiceAccounts, RBAC
```

**SSL/TLS with Cert-Manager:**

Automatically provision Let's Encrypt certificates:

```bash
# Install cert-manager
kubectl apply -f https://github.com/cert-manager/cert-manager/releases/download/v1.13.0/cert-manager.yaml

# Create ClusterIssuer
kubectl apply -f - <<EOF
apiVersion: cert-manager.io/v1
kind: ClusterIssuer
metadata:
  name: letsencrypt-prod
spec:
  acme:
    server: https://acme-v02.api.letsencrypt.org/directory
    email: your@email.com
    privateKeySecretRef:
      name: letsencrypt-prod
    solvers:
    - http01:
        ingress:
          class: nginx
EOF
```

Now just add annotation to Ingress:
```yaml
annotations:
  cert-manager.io/cluster-issuer: "letsencrypt-prod"
```

Cert-manager automatically:
1. Detects Ingress with annotation
2. Requests certificate from Let's Encrypt
3. Solves ACME challenge
4. Stores certificate in Secret
5. Configures Ingress to use it
6. Renews before expiration

### ConfigMap - Configuration Data

**Problem:** Hard-coded configuration in images:

```python
DATABASE_URL = "postgresql://db:5432/mydb"
LOG_LEVEL = "INFO"
```

Change requires rebuilding image. Different environments (dev/prod) need different values.

**Solution: ConfigMap**

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config
data:
  # Key-value pairs
  database_host: "postgres-service"
  database_port: "5432"
  database_name: "mydb"
  log_level: "INFO"
  
  # Or entire config files
  app.conf: |
    [database]
    host = postgres-service
    port = 5432
    
    [logging]
    level = INFO
  
  config.json: |
    {
      "database": {
        "host": "postgres-service",
        "port": 5432
      },
      "features": {
        "new_ui": true
      }
    }
```

**Using ConfigMap - Environment Variables:**

```yaml
spec:
  containers:
  - name: myapp
    image: myapp:v1
    env:
    # Single key
    - name: DATABASE_HOST
      valueFrom:
        configMapKeyRef:
          name: app-config
          key: database_host
    
    # All keys as environment variables
    envFrom:
    - configMapRef:
        name: app-config
    # Creates: DATABASE_HOST, DATABASE_PORT, DATABASE_NAME, LOG_LEVEL
```

**Using ConfigMap - Volume Mount:**

```yaml
spec:
  containers:
  - name: myapp
    image: myapp:v1
    volumeMounts:
    - name: config
      mountPath: /etc/config
      # All ConfigMap keys become files in /etc/config
  
  volumes:
  - name: config
    configMap:
      name: app-config

# Results in:
# /etc/config/database_host (contains: postgres-service)
# /etc/config/database_port (contains: 5432)
# /etc/config/app.conf (contains: full config file)
# /etc/config/config.json (contains: full JSON)
```

**Updating ConfigMap:**

```bash
# Edit ConfigMap
kubectl edit configmap app-config

# Or apply new version
kubectl apply -f configmap.yaml
```

**Important:** Pods using ConfigMap as environment variables don't see changes (must restart pod). Pods using ConfigMap as volume mount see changes after ~60 seconds (automatically).

### Secret - Sensitive Data

**Similar to ConfigMap but for sensitive data:**
- Passwords
- API keys
- Tokens
- Certificates

```yaml
apiVersion: v1
kind: Secret
metadata:
  name: db-secret
type: Opaque
data:
  # Values must be base64 encoded
  username: cG9zdGdyZXM=  # postgres
  password: c2VjcmV0MTIz  # secret123
```

**Create Secret from Command:**

```bash
kubectl create secret generic db-secret \
  --from-literal=username=postgres \
  --from-literal=password=secret123

# Or from files
kubectl create secret generic tls-secret \
  --from-file=tls.crt \
  --from-file=tls.key
```

**Using Secrets:**

```yaml
spec:
  containers:
  - name: myapp
    image: myapp:v1
    env:
    - name: DB_USERNAME
      valueFrom:
        secretKeyRef:
          name: db-secret
          key: username
    - name: DB_PASSWORD
      valueFrom:
        secretKeyRef:
          name: db-secret
          key: password
    
    # Or mount as volume
    volumeMounts:
    - name: secrets
      mountPath: /etc/secrets
      readOnly: true
  
  volumes:
  - name: secrets
    secret:
      secretName: db-secret
```

**Important Security Notes:**

1. Secrets are **base64 encoded, NOT encrypted** by default
2. Stored in etcd (encrypt etcd for production!)
3. Only sent to nodes running pods that use them
4. Stored in tmpfs (memory, not disk)
5. RBAC controls who can read secrets

**For production:**
- Enable etcd encryption at rest
- Use external secret management (HashiCorp Vault, AWS Secrets Manager)
- Rotate secrets regularly
- Use RBAC to restrict access

### StatefulSet - Stateful Applications

**Deployment vs StatefulSet:**

|Deployment|StatefulSet|
|----------|-----------|
|Pods are interchangeable|Each pod has unique identity|
|Random names: myapp-7d8f9-abc12|Ordered names: myapp-0, myapp-1, myapp-2|
|Any pod can be killed first|Pods created/deleted in order|
|No stable network identity|Stable hostname per pod|
|No persistent storage per pod|Persistent storage follows pod|

**Use StatefulSet for:**
- Databases (PostgreSQL, MySQL, MongoDB)
- Distributed systems (Kafka, Zookeeper, Cassandra)
- Applications needing stable hostnames

```yaml
apiVersion: v1
kind: Service
metadata:
  name: postgres-headless
spec:
  clusterIP: None
  # Headless service for StatefulSet
  selector:
    app: postgres
  ports:
  - port: 5432
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: postgres
spec:
  serviceName: postgres-headless
  # Required: name of headless service
  
  replicas: 3
  
  selector:
    matchLabels:
      app: postgres
  
  template:
    metadata:
      labels:
        app: postgres
    spec:
      containers:
      - name: postgres
        image: postgres:15
        ports:
        - containerPort: 5432
        env:
        - name: POSTGRES_PASSWORD
          value: secret
        volumeMounts:
        - name: data
          mountPath: /var/lib/postgresql/data
  
  volumeClaimTemplates:
  # Each pod gets its own PVC (PersistentVolumeClaim)
  - metadata:
      name: data
    spec:
      accessModes: ["ReadWriteOnce"]
      resources:
        requests:
          storage: 10Gi
```

**What Happens:**

```
1. Create StatefulSet with replicas=3

2. Pods created in order:
   - postgres-0 (waits until Running)
   - postgres-1 (waits until Running)
   - postgres-2

3. Each pod gets:
   - Stable hostname: postgres-0.postgres-headless.default.svc.cluster.local
   - Persistent Volume: data-postgres-0, data-postgres-1, data-postgres-2

4. DNS entries:
   postgres-0.postgres-headless → IP of postgres-0
   postgres-1.postgres-headless → IP of postgres-1
   postgres-2.postgres-headless → IP of postgres-2

5. If postgres-1 dies:
   - New postgres-1 created on any node
   - Gets same hostname
   - Reattaches to data-postgres-1 volume
   - Data persists!
```

**Ordered Operations:**

**Scaling up (3 → 5):**
```
postgres-3 created → Running → postgres-4 created
```

**Scaling down (5 → 3):**
```
postgres-4 deleted → Termination complete → postgres-3 deleted
```

**Updating:**
```
RollingUpdate (default):
postgres-2 updated → Running → postgres-1 updated → Running → postgres-0 updated

OnDelete:
Only updates when pod is manually deleted
```

### DaemonSet - One Pod Per Node

**Purpose:** Ensure every node runs a copy of a pod.

**Use Cases:**
- Log collection (Fluentd on every node)
- Monitoring (Prometheus Node Exporter)
- Network plugins (Calico, Weave)
- Storage daemons

```yaml
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: fluentd
spec:
  selector:
    matchLabels:
      name: fluentd
  template:
    metadata:
      labels:
        name: fluentd
    spec:
      containers:
      - name: fluentd
        image: fluentd:v1.14
        volumeMounts:
        - name: varlog
          mountPath: /var/log
          # Access to node's /var/log
        - name: varlibdockercontainers
          mountPath: /var/lib/docker/containers
          readOnly: true
          # Access to container logs
      
      volumes:
      - name: varlog
        hostPath:
          path: /var/log
      - name: varlibdockercontainers
        hostPath:
          path: /var/lib/docker/containers
```

**Behavior:**
- When new node added to cluster, DaemonSet automatically adds pod
- When node removed, pod is garbage collected
- Deleting DaemonSet removes all its pods

**Node Selection:**

Run on specific nodes only:

```yaml
spec:
  template:
    spec:
      nodeSelector:
        disktype: ssd
        # Only nodes with label disktype=ssd
```

### Job - Run-to-Completion

**For batch processing, one-time tasks:**

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: data-migration
spec:
  completions: 5
  # Job completes when 5 pods succeed
  
  parallelism: 2
  # Run 2 pods in parallel
  
  backoffLimit: 3
  # Retry 3 times if pod fails
  
  activeDeadlineSeconds: 600
  # Job fails if not complete in 10 minutes
  
  template:
    spec:
      containers:
      - name: migrator
        image: data-migrator:v1
        command: ["python", "migrate.py"]
      
      restartPolicy: OnFailure
      # Never or OnFailure (not Always!)
```

**Execution:**

```
Start: 0/5 complete
  ↓
Create pod 1, pod 2 (parallelism=2)
  ↓
Pod 1 succeeds → 1/5 complete
Pod 2 succeeds → 2/5 complete
  ↓
Create pod 3, pod 4
  ↓
Pod 3 fails → restart (backoffLimit)
Pod 4 succeeds → 3/5 complete
  ↓
Pod 3 succeeds → 4/5 complete
  ↓
Create pod 5
  ↓
Pod 5 succeeds → 5/5 complete
  ↓
Job Complete
```

**Use Cases:**
- Database migrations
- ETL jobs
- Batch processing
- Report generation
- ML training

### CronJob - Scheduled Jobs

**Like cron, but for Kubernetes:**

```yaml
apiVersion: batch/v1
kind: CronJob
metadata:
  name: daily-backup
spec:
  schedule: "0 2 * * *"
  # Cron format: minute hour day month dayofweek
  # This runs at 2:00 AM daily
  
  successfulJobsHistoryLimit: 3
  # Keep last 3 successful jobs
  failedJobsHistoryLimit: 1
  # Keep last 1 failed job
  
  jobTemplate:
    # Job spec (same as Job)
    spec:
      template:
        spec:
          containers:
          - name: backup
            image: backup-tool:v1
            command: ["python", "backup.py"]
          restartPolicy: OnFailure
```

**Cron Schedule Examples:**

```
"*/5 * * * *"     # Every 5 minutes
"0 */2 * * *"     # Every 2 hours
"0 0 * * *"       # Daily at midnight
"0 0 * * 0"       # Weekly on Sunday
"0 0 1 * *"       # Monthly on 1st
"0 9 * * 1-5"     # Weekdays at 9 AM
```

**Use Cases:**
- Daily database backups
- Periodic data cleanup
- Report generation
- Health checks
- Cache warming

---



## Kubernetes Networking {#k8s-networking}

### The Kubernetes Network Model

Kubernetes has unique networking requirements compared to Docker:

**Requirements:**
1. All pods can communicate with all other pods without NAT
2. All nodes can communicate with all pods without NAT
3. IP a pod sees itself as is the same IP others see it as

This creates a flat network space where:
- Pod on Node A can directly reach Pod on Node B
- No port conflicts (each pod has unique IP)
- Simplified networking (no port mapping hell)

**Network Ranges:**

```
Cluster CIDR: 10.0.0.0/16
├── Node 1 Pod CIDR: 10.1.0.0/24
│   ├── Pod A: 10.1.0.2
│   ├── Pod B: 10.1.0.3
│   └── Pod C: 10.1.0.4
├── Node 2 Pod CIDR: 10.1.1.0/24
│   ├── Pod D: 10.1.1.2
│   └── Pod E: 10.1.1.3
└── Service CIDR: 10.96.0.0/12
    ├── Service A: 10.96.0.1
    └── Service B: 10.96.0.2
```

### Pod-to-Pod Communication

**Same Node:**

```
Pod A (10.1.0.2) wants to reach Pod B (10.1.0.3)

Pod A → veth0 (virtual ethernet) → 
cbr0 bridge (like docker0) → 
veth1 → Pod B

Direct communication, no routing needed.
```

**Different Nodes:**

```
Pod A (10.1.0.2) on Node 1 wants to reach Pod D (10.1.1.2) on Node 2

Pod A → veth0 → cbr0 → Node 1 network →
Overlay Network (VXLAN/IP-in-IP/etc.) →
Node 2 network → cbr0 → veth0 → Pod D

CNI plugin handles this routing.
```

**Container Network Interface (CNI):**

Kubernetes doesn't implement networking itself. It uses CNI plugins:

**Popular CNI Plugins:**

**Calico:**
- Uses BGP routing
- Supports Network Policies
- High performance
- No overlay (routes directly)

**Flannel:**
- Simple, easy to set up
- VXLAN overlay
- No Network Policy support

**Cilium:**
- eBPF-based
- Advanced security features
- Service mesh capabilities
- Great performance

**Weave:**
- Mesh network
- Encryption support
- Simple setup

### Service Networking Deep Dive

**How Services Actually Work:**

```yaml
apiVersion: v1
kind: Service
metadata:
  name: api-service
spec:
  selector:
    app: api
  ports:
  - port: 80
    targetPort: 8000
```

**Behind the Scenes:**

```
1. Service created with ClusterIP: 10.96.0.10

2. Endpoints Controller finds pods with label app=api:
   - Pod A: 10.1.0.2:8000
   - Pod B: 10.1.0.3:8000
   - Pod C: 10.1.1.2:8000

3. Creates Endpoints object:
   api-service → [10.1.0.2:8000, 10.1.0.3:8000, 10.1.1.2:8000]

4. kube-proxy on EVERY node configures iptables/IPVS:
   - Traffic to 10.96.0.10:80 →
   - Randomly select one endpoint →
   - DNAT to selected pod IP:port
```

**iptables Mode (Default):**

kube-proxy creates iptables rules:

```bash
# Simplified view
iptables -A KUBE-SERVICES -d 10.96.0.10/32 -p tcp --dport 80 -j KUBE-SVC-API

# KUBE-SVC-API chain:
-A KUBE-SVC-API -m statistic --mode random --probability 0.33 -j KUBE-SEP-1
-A KUBE-SVC-API -m statistic --mode random --probability 0.50 -j KUBE-SEP-2
-A KUBE-SVC-API -j KUBE-SEP-3

# Each KUBE-SEP chain DNATs to pod:
-A KUBE-SEP-1 -p tcp -j DNAT --to-destination 10.1.0.2:8000
-A KUBE-SEP-2 -p tcp -j DNAT --to-destination 10.1.0.3:8000
-A KUBE-SEP-3 -p tcp -j DNAT --to-destination 10.1.1.2:8000
```

Random load balancing via probability matching.

**IPVS Mode (Better for Scale):**

```bash
# Enable IPVS mode
kube-proxy --proxy-mode=ipvs
```

Uses Linux IPVS instead of iptables.

Benefits:
- Better performance at scale (thousands of services)
- More load balancing algorithms (rr, lc, dh, sh, etc.)
- Deterministic behavior

**Service Discovery - DNS:**

CoreDNS runs in kube-system namespace.

```
Pod wants to connect to "api-service"
  ↓
Pod's DNS resolver: 10.96.0.10 (kube-dns service)
  ↓
Query: api-service.default.svc.cluster.local
  ↓
CoreDNS responds: 10.96.0.10 (Service ClusterIP)
  ↓
Pod connects to 10.96.0.10
  ↓
kube-proxy routes to pod
```

**DNS Naming:**

```
<service>.<namespace>.svc.cluster.local

Examples:
api-service.default.svc.cluster.local
postgres.production.svc.cluster.local
redis.cache.svc.cluster.local

Short forms work within same namespace:
api-service
api-service.default
```

### Network Policies - Pod Firewall

**Default Behavior:**

By default, all pods can communicate with all pods. No restrictions.

**Problem:**

Frontend pods can reach database pods directly (bad security).

**Solution: NetworkPolicy**

```yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: api-network-policy
  namespace: default
spec:
  podSelector:
    matchLabels:
      app: api
  # Apply policy to pods with label app=api
  
  policyTypes:
  - Ingress
  - Egress
  
  ingress:
  # Allow incoming traffic from:
  - from:
    - podSelector:
        matchLabels:
          app: frontend
    # Only from pods labeled app=frontend
    ports:
    - protocol: TCP
      port: 8000
  
  egress:
  # Allow outgoing traffic to:
  - to:
    - podSelector:
        matchLabels:
          app: database
    ports:
    - protocol: TCP
      port: 5432
  
  - to:
    # Allow DNS
    - namespaceSelector:
        matchLabels:
          name: kube-system
    ports:
    - protocol: UDP
      port: 53
```

**Effect:**

```
Before NetworkPolicy:
Frontend → API ✓
Frontend → Database ✓ (bad!)
Random Pod → API ✓ (bad!)

After NetworkPolicy:
Frontend → API ✓ (allowed by ingress rule)
Frontend → Database ✗ (no policy allows this)
Random Pod → API ✗ (not labeled app=frontend)
API → Database ✓ (allowed by egress rule)
```

**Deny All Policy:**

```yaml
# Deny all ingress
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: deny-all
spec:
  podSelector: {}
  # Empty selector = all pods
  policyTypes:
  - Ingress
  # No ingress rules = deny all
```

**Namespace Isolation:**

```yaml
# Only allow traffic within same namespace
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: namespace-isolation
spec:
  podSelector: {}
  policyTypes:
  - Ingress
  ingress:
  - from:
    - podSelector: {}
    # Empty podSelector = all pods in this namespace
```

**Important:**

Network Policies require CNI plugin support:
- Calico: ✓
- Cilium: ✓
- Weave: ✓
- Flannel: ✗ (no Network Policy support)

---



## Storage in Kubernetes 

### The Storage Problem

Containers are ephemeral. Data inside container is lost when container dies.

```
Pod writes data to /app/data
  ↓
Pod crashes
  ↓
New pod created
  ↓
/app/data is empty (data lost!)
```

Volumes solve this problem.

### Volume Types

**emptyDir - Temporary Storage:**

```yaml
spec:
  containers:
  - name: myapp
    volumeMounts:
    - name: cache
      mountPath: /cache
  volumes:
  - name: cache
    emptyDir: {}
```

- Created when pod starts
- Deleted when pod deleted
- Shared between containers in pod
- Stored on node's disk (or memory with `emptyDir: {medium: Memory}`)

**Use case:** Scratch space, cache, temporary files.

**hostPath - Node Storage:**

```yaml
volumes:
- name: logs
  hostPath:
    path: /var/log/pods
    type: Directory
```

- Mounts directory from node's filesystem
- Data persists even if pod deleted
- Pod always scheduled to same node to access data

**Dangers:**
- Pod on different node can't access data
- Security risk (pod accesses host filesystem)
- Don't use in production (unless you know what you're doing)

**Use case:** System daemons (DaemonSets), accessing node logs.

### Persistent Volumes (PV) and Claims (PVC)

**The Abstraction:**

```
Developer: "I need 10GB storage"
  ↓
Kubernetes: "Here's a volume"
  ↓
Developer uses it, doesn't care about underlying storage
```

**PersistentVolume (PV)** - Cluster resource, represents physical storage.
**PersistentVolumeClaim (PVC)** - Request for storage by user.

**Flow:**

```
1. Admin creates PV (or StorageClass provisions automatically)
2. User creates PVC requesting storage
3. Kubernetes binds PVC to suitable PV
4. Pod uses PVC
```

**PersistentVolume:**

```yaml
apiVersion: v1
kind: PersistentVolume
metadata:
  name: pv-data
spec:
  capacity:
    storage: 10Gi
  # Total capacity
  
  accessModes:
  - ReadWriteOnce
  # RWO: Single node read-write
  # ROX: Multiple nodes read-only
  # RWX: Multiple nodes read-write
  
  persistentVolumeReclaimPolicy: Retain
  # Retain: Manual reclamation after claim deleted
  # Delete: Automatically delete storage
  # Recycle: Deprecated
  
  storageClassName: standard
  # Group PVs into classes
  
  hostPath:
    path: /mnt/data
  # For development only!
  # Production uses cloud storage:
  # - awsElasticBlockStore
  # - gcePersistentDisk
  # - azureDisk
  # - nfs
  # - cephfs
  # - etc.
```

**PersistentVolumeClaim:**

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: data-pvc
spec:
  accessModes:
  - ReadWriteOnce
  # Must match PV's accessModes
  
  resources:
    requests:
      storage: 5Gi
  # Request 5Gi (PV has 10Gi, that's fine)
  
  storageClassName: standard
  # Must match PV's storageClassName
```

**Binding:**

```
PVC created: 5Gi, RWO, standard
  ↓
Kubernetes finds PV:
  - Has >= 5Gi? ✓ (10Gi)
  - AccessMode matches? ✓ (RWO)
  - StorageClass matches? ✓ (standard)
  ↓
Bind PVC to PV
  ↓
PVC Status: Bound
PV Status: Bound
```

**Using PVC in Pod:**

```yaml
spec:
  containers:
  - name: myapp
    volumeMounts:
    - name: data
      mountPath: /app/data
  volumes:
  - name: data
    persistentVolumeClaim:
      claimName: data-pvc
```

**Lifecycle:**

```
1. PV and PVC created
2. PVC bound to PV
3. Pod uses PVC
4. Pod deleted → PVC still exists, data persists
5. PVC deleted → PV status: Released
6. If Retain policy:
   - PV not deleted
   - Admin must manually clean and make Available
7. If Delete policy:
   - PV and underlying storage automatically deleted
```

### StorageClass - Dynamic Provisioning

**Problem:** Admin must manually create PVs. Tedious!

**Solution:** StorageClass automatically creates PVs on demand.

```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: fast-ssd
provisioner: kubernetes.io/aws-ebs
# Provisioner: creates actual storage
# Different for each cloud:
# - kubernetes.io/aws-ebs (AWS)
# - kubernetes.io/gce-pd (GCP)
# - kubernetes.io/azure-disk (Azure)
# - k8s.io/minikube-hostpath (Minikube)

parameters:
  type: gp3
  # AWS EBS volume type
  iopsPerGB: "50"
  encrypted: "true"

allowVolumeExpansion: true
# Can expand PVC size later

volumeBindingMode: WaitForFirstConsumer
# Two modes:
# - Immediate: Provision storage as soon as PVC created
# - WaitForFirstConsumer: Wait until pod using PVC is scheduled
#   (Ensures storage in same zone as pod)

reclaimPolicy: Delete
# Delete or Retain
```

**Dynamic Provisioning Flow:**

```
1. User creates PVC with storageClassName: fast-ssd

2. No existing PV matches (Size/AccessMode/Class)
  ↓
3. StorageClass controller sees unfulfilled PVC
  ↓
4. Calls AWS API: create EBS volume (gp3, 100GB, encrypted)
  ↓
5. AWS creates EBS volume
  ↓
6. Controller creates PV object pointing to EBS volume
  ↓
7. Binds PVC to new PV
  ↓
8. Pod can now use PVC
```

**PVC with StorageClass:**

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: data-pvc
spec:
  accessModes:
  - ReadWriteOnce
  storageClassName: fast-ssd
  # Uses StorageClass to provision
  resources:
    requests:
      storage: 100Gi
```

No need to create PV manually! StorageClass does it automatically.

**Volume Expansion:**

```yaml
# Original PVC: 100Gi
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: data-pvc
spec:
  resources:
    requests:
      storage: 200Gi
  # Changed to 200Gi

# Apply:
kubectl apply -f pvc.yaml

# StorageClass expands underlying storage
# Pod may need restart to see new size
```

### StatefulSet Storage

**Each pod gets its own persistent volume:**

```yaml
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: mysql
spec:
  serviceName: mysql-headless
  replicas: 3
  selector:
    matchLabels:
      app: mysql
  template:
    metadata:
      labels:
        app: mysql
    spec:
      containers:
      - name: mysql
        image: mysql:8.0
        volumeMounts:
        - name: data
          mountPath: /var/lib/mysql
  
  volumeClaimTemplates:
  # Template for creating PVCs
  - metadata:
      name: data
    spec:
      accessModes: ["ReadWriteOnce"]
      storageClassName: fast-ssd
      resources:
        requests:
          storage: 50Gi
```

**What Happens:**

```
StatefulSet created with replicas=3
  ↓
Pod mysql-0 created
  ↓
PVC data-mysql-0 created from template
  ↓
StorageClass provisions PV (50Gi EBS)
  ↓
PVC bound to PV
  ↓
Pod mysql-0 starts, mounts data-mysql-0
  ↓
Pod mysql-1 created
  ↓
PVC data-mysql-1 created
  ↓
(Repeat for mysql-2)

Result:
mysql-0 → data-mysql-0 (PV 1)
mysql-1 → data-mysql-1 (PV 2)
mysql-2 → data-mysql-2 (PV 3)
```

**If pod deleted:**

```
kubectl delete pod mysql-1
  ↓
New mysql-1 created (possibly on different node)
  ↓
Reattaches to data-mysql-1
  ↓
Data persists!
```

**Scaling:**

```
Scale up to 5:
  ↓
Creates mysql-3 with data-mysql-3
Creates mysql-4 with data-mysql-4

Scale down to 2:
  ↓
Deletes mysql-4, mysql-3
BUT: data-mysql-3 and data-mysql-4 remain!

Scale back up to 4:
  ↓
Recreates mysql-3, reattaches to existing data-mysql-3
Recreates mysql-4, reattaches to existing data-mysql-4
  ↓
Data preserved!
```

---



## Autoscaling & Resource Management

### Resource Requests and Limits

**Why They Matter:**

Without resource limits:
- One pod can starve others (resource hogging)
- No fair scheduling (pods randomly placed)
- OOM kills unpredictable (which pod gets killed?)

**Requests vs Limits:**

```yaml
resources:
  requests:
    memory: "256Mi"
    cpu: "250m"
  # Requests: Minimum guaranteed
  # Scheduler uses this for placement
  
  limits:
    memory: "512Mi"
    cpu: "500m"
  # Limits: Maximum allowed
```

**How It Works:**

**CPU (Compressible Resource):**

```
Pod requests 250m (0.25 cores)
Scheduled to node with 1 core available
  ↓
Pod runs, uses 250m
  ↓
Pod tries to use 600m (exceeds limit of 500m)
  ↓
Kernel throttles CPU to 500m
  ↓
Pod slows down but keeps running
```

CPU is throttled when limit exceeded. Pod not killed.

**Memory (Non-Compressible Resource):**

```
Pod requests 256Mi
Scheduled to node with 1Gi available
  ↓
Pod runs, uses 256Mi
  ↓
Pod allocates more memory, reaches 512Mi (limit)
  ↓
Pod tries to allocate more
  ↓
OOMKilled! (Out of Memory)
  ↓
kubelet restarts container
```

Memory can't be throttled. Exceeding limit = death.

**CPU Units:**

```
1 CPU = 1000m (millicores)
1000m = 1 AWS vCPU = 1 GCP Core = 1 Azure vCore = 1 hyperthread

Examples:
500m = 0.5 CPU
250m = 0.25 CPU (25% of one core)
2000m = 2 CPUs
100m = 0.1 CPU (minimum reasonable value)
```

**Memory Units:**

```
Ki = Kibibyte = 1024 bytes
Mi = Mebibyte = 1024 Ki
Gi = Gibibyte = 1024 Mi

Examples:
128Mi = 128 mebibytes = 134 megabytes
1Gi = 1 gibibyte = 1.07 gigabytes
512Mi = 0.5 gibibytes
```

**Quality of Service (QoS) Classes:**

Based on requests and limits, pods get QoS class:

**1. Guaranteed (Highest Priority):**

```yaml
resources:
  requests:
    memory: "256Mi"
    cpu: "250m"
  limits:
    memory: "256Mi"
    cpu: "250m"
# requests == limits
```

- Highest priority during eviction
- Won't be killed unless exceeds limits
- Use for critical workloads

**2. Burstable (Medium Priority):**

```yaml
resources:
  requests:
    memory: "256Mi"
    cpu: "250m"
  limits:
    memory: "512Mi"
    cpu: "500m"
# requests < limits
```

- Medium priority
- Can burst above requests (if resources available)
- Killed if node runs out of memory (after BestEffort)
- Use for most workloads

**3. BestEffort (Lowest Priority):**

```yaml
# No requests or limits
resources: {}
```

- Lowest priority
- First to be killed during node pressure
- Can use any available resources
- Use for non-critical batch jobs

**Eviction Order (when node runs out of resources):**

```
1. BestEffort pods killed first
2. Burstable pods exceeding requests
3. Burstable pods within requests
4. Guaranteed pods (only if using > limits)
```

### Horizontal Pod Autoscaler (HPA)

**Automatically scale number of pods based on metrics.**

```yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: myapp-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: myapp-deployment
  # What to scale
  
  minReplicas: 2
  # Minimum pods (even if usage low)
  
  maxReplicas: 10
  # Maximum pods (even if usage high)
  
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
  # Target: 70% CPU utilization across all pods
  
  - type: Resource
    resource:
      name: memory
      target:
        type: Utilization
        averageUtilization: 80
  # Target: 80% memory utilization
  
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 300
      # Wait 5 min after metrics drop before scaling down
      # Prevents flapping
      
      policies:
      - type: Percent
        value: 50
        periodSeconds: 60
      # Scale down max 50% of current pods per minute
      
      - type: Pods
        value: 2
        periodSeconds: 60
      # OR scale down max 2 pods per minute
      # (Whichever is smaller)
    
    scaleUp:
      stabilizationWindowSeconds: 0
      # Scale up immediately (no delay)
      
      policies:
      - type: Percent
        value: 100
        periodSeconds: 15
      # Scale up max 100% every 15 seconds
      # Can double pods quickly if needed
      
      - type: Pods
        value: 4
        periodSeconds: 15
      # OR add max 4 pods every 15 seconds
```

**How HPA Works:**

```
Every 15 seconds (default):

1. HPA Controller queries Metrics Server
   
2. Gets current CPU utilization:
   Pod 1: 80%
   Pod 2: 85%
   Pod 3: 75%
   Average: 80%

3. Compares to target: 70%
   Current: 80%
   Desired: 70%

4. Calculates desired replicas:
   desiredReplicas = ceil[currentReplicas * (currentMetric / targetMetric)]
   desiredReplicas = ceil[3 * (80 / 70)]
   desiredReplicas = ceil[3.43]
   desiredReplicas = 4

5. Checks constraints:
   - Within min/max? ✓ (4 is between 2 and 10)
   - Behavior policies? ✓

6. Updates Deployment: replicas = 4

7. Deployment Controller creates 1 new pod

8. Wait for pod to be ready, start serving traffic

9. Repeat in 15 seconds
```

**Example Scenario:**

```
Initial: 2 pods, 40% CPU
  ↓ Load increases
Current: 2 pods, 90% CPU (above target 70%)
  ↓
HPA: Scale to 3 pods (ceil[2 * (90/70)] = 3)
  ↓
Current: 3 pods, 60% CPU (below target)
  ↓ Nothing (within stabilization window)
After 5 min: Still 60% CPU
  ↓
HPA: Can scale down now (stabilization window passed)
  ↓
HPA: Scale to 2 pods (ceil[3 * (60/70)] = 3, no change)
  ↓ Load decreases further
Current: 3 pods, 40% CPU
  ↓
HPA: Scale to 2 pods (ceil[3 * (40/70)] = 2)
```

**Prerequisites:**

HPA requires Metrics Server:

```bash
# Install Metrics Server
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

# Verify
kubectl get apiservices | grep metrics

# View metrics
kubectl top nodes
kubectl top pods
```

**Custom Metrics:**

Scale on metrics beyond CPU/memory:

```yaml
metrics:
- type: Pods
  pods:
    metric:
      name: http_requests_per_second
    target:
      type: AverageValue
      averageValue: "1000"
# Scale to keep 1000 requests/second per pod

- type: Object
  object:
    metric:
      name: requests-per-second
    describedObject:
      apiVersion: networking.k8s.io/v1
      kind: Ingress
      name: myapp-ingress
    target:
      type: Value
      value: "10k"

- type: External
  external:
    metric:
      name: queue_messages
      selector:
        matchLabels:
          queue: "jobs"
    target:
      type: Value
      value: "30"
# Scale to keep queue at 30 messages
```

Requires custom metrics adapter:
- Prometheus Adapter (most common)
- Datadog Cluster Agent
- Google Cloud Monitoring
- etc.

### Vertical Pod Autoscaler (VPA)

**Automatically adjust resource requests/limits.**

**Use Case:**

You guessed:
- Pod requests 256Mi, actually needs 1Gi (underprovisioned, gets OOMKilled)
- Pod requests 4Gi, actually uses 512Mi (overprovisioned, wastes resources)

VPA analyzes usage and recommends/applies correct values.

```yaml
apiVersion: autoscaling.k8s.io/v1
kind: VerticalPodAutoscaler
metadata:
  name: myapp-vpa
spec:
  targetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: myapp-deployment
  
  updatePolicy:
    updateMode: "Auto"
    # Modes:
    # - "Off": Only recommendations, no changes
    # - "Initial": Set resources on pod creation only
    # - "Recreate": Update existing pods (requires restart)
    # - "Auto": Currently same as Recreate
  
  resourcePolicy:
    containerPolicies:
    - containerName: myapp
      minAllowed:
        cpu: 100m
        memory: 128Mi
      # VPA won't set resources below this
      
      maxAllowed:
        cpu: 2000m
        memory: 2Gi
      # VPA won't set resources above this
      
      controlledResources:
      - cpu
      - memory
      # Which resources VPA controls
```

**How VPA Works:**

```
1. VPA Recommender watches pod metrics (7 days by default)

2. Analyzes usage patterns:
   Peak CPU: 650m
   95th percentile CPU: 450m
   Average CPU: 300m
   
   Peak Memory: 800Mi
   95th percentile Memory: 600Mi
   Average Memory: 400Mi

3. Calculates recommendations:
   CPU request: 450m (95th percentile)
   CPU limit: 900m (2x request)
   Memory request: 600Mi (95th percentile)
   Memory limit: 1200Mi (2x request)

4. If updateMode=Auto:
   - VPA Updater evicts pod
   - New pod created with new resources
   - (Requires PodDisruptionBudget for high availability!)

5. Pod runs with right-sized resources
```

**Important Notes:**

- **VPA and HPA conflict:** Both scaling at once causes issues. Don't use both on same metric.
- **VPA requires pod restart:** Changing resources needs new pod (can't change running pod)
- **Use PodDisruptionBudget:** Prevent all pods from being evicted at once

**Checking VPA Recommendations:**

```bash
kubectl describe vpa myapp-vpa

# Shows:
# Recommendation:
#   Container myapp:
#     CPU:
#       Lower Bound: 100m
#       Target: 450m
#       Upper Bound: 900m
#     Memory:
#       Lower Bound: 128Mi
#       Target: 600Mi
#       Upper Bound: 1200Mi
```

### Cluster Autoscaler

**Automatically add/remove nodes based on pending pods.**

**Scenario:**

```
20 pods need scheduling
Current nodes fully utilized
  ↓
Pods stuck in Pending state
  ↓
Cluster Autoscaler detects pending pods
  ↓
Calls cloud API: create new node
  ↓
New node joins cluster
  ↓
Scheduler places pending pods on new node
```

**Installation (Cloud-specific):**

```yaml
# AWS example
apiVersion: apps/v1
kind: Deployment
metadata:
  name: cluster-autoscaler
  namespace: kube-system
spec:
  selector:
    matchLabels:
      app: cluster-autoscaler
  template:
    metadata:
      labels:
        app: cluster-autoscaler
    spec:
      serviceAccountName: cluster-autoscaler
      containers:
      - name: cluster-autoscaler
        image: k8s.gcr.io/autoscaling/cluster-autoscaler:v1.27.0
        command:
        - ./cluster-autoscaler
        - --v=4
        - --stderrthreshold=info
        - --cloud-provider=aws
        - --skip-nodes-with-local-storage=false
        - --nodes=2:10:my-nodegroup
        # min:max:nodegroup-name
        - --scale-down-delay-after-add=10m
        # Wait 10 min after scale-up before considering scale-down
        - --scale-down-unneeded-time=10m
        # Node must be unneeded for 10 min before removal
```

**Scale Up Conditions:**

- Pods are pending due to insufficient resources (CPU, memory, GPU)
- Adding a node would allow pending pods to schedule
- Not at maximum node count
- No PodDisruptionBudget violations

**Scale Down Conditions:**

- Node utilization < 50% (by default)
- All pods on node can be moved to other nodes
- No pods with local storage (unless --skip-nodes-with-local-storage=false)
- No system pods (kube-system) without PodDisruptionBudget
- Node unneeded for > 10 minutes (configurable)

**Preventing Scale Down:**

```yaml
# Annotation on node
metadata:
  annotations:
    cluster-autoscaler.kubernetes.io/scale-down-disabled: "true"

# Or pod annotation (prevent node with this pod from being removed)
metadata:
  annotations:
    cluster-autoscaler.kubernetes.io/safe-to-evict: "false"
```

### Resource Quotas - Namespace Limits

**Prevent one team from using all cluster resources:**

```yaml
apiVersion: v1
kind: ResourceQuota
metadata:
  name: compute-quota
  namespace: development
spec:
  hard:
    requests.cpu: "10"
    # Total CPU requests across all pods in namespace
    requests.memory: "20Gi"
    # Total memory requests
    
    limits.cpu: "20"
    # Total CPU limits
    limits.memory: "40Gi"
    # Total memory limits
    
    pods: "50"
    # Max number of pods
    
    services: "20"
    # Max number of services
    
    persistentvolumeclaims: "10"
    # Max number of PVCs
    
    requests.storage: "100Gi"
    # Total storage requests
```

**Effect:**

```
Development namespace has quota:
  requests.cpu: 10
  Currently used: 8

User creates deployment: requests.cpu: 250m × 10 pods = 2.5 cores
Total would be: 8 + 2.5 = 10.5
  ↓
Rejected! "exceeded quota"

User reduces to 8 pods = 2 cores
Total would be: 8 + 2 = 10
  ↓
Accepted! ✓
```

**Viewing Quota:**

```bash
kubectl get resourcequota -n development
kubectl describe resourcequota compute-quota -n development

# Shows:
# Name: compute-quota
# Resource       Used   Hard
# --------       ----   ----
# limits.cpu     15     20
# limits.memory  30Gi   40Gi
# pods           35     50
# requests.cpu   8      10
# requests.memory 16Gi  20Gi
```

### LimitRange - Default and Min/Max Resources

**Set default resources and enforce limits:**

```yaml
apiVersion: v1
kind: LimitRange
metadata:
  name: resource-limits
  namespace: development
spec:
  limits:
  - max:
      cpu: "2"
      memory: "4Gi"
    # Maximum per pod/container
    
    min:
      cpu: "100m"
      memory: "128Mi"
    # Minimum per pod/container
    
    default:
      cpu: "500m"
      memory: "512Mi"
    # Default limits (if not specified)
    
    defaultRequest:
      cpu: "250m"
      memory: "256Mi"
    # Default requests (if not specified)
    
    type: Container
    # Apply to containers (also: Pod, PVC)
```

**Effect:**

```yaml
# User creates pod without resources:
spec:
  containers:
  - name: myapp
    image: myapp:v1
    # No resources specified

# LimitRange injects defaults:
spec:
  containers:
  - name: myapp
    image: myapp:v1
    resources:
      requests:
        cpu: "250m"
        memory: "256Mi"
      limits:
        cpu: "500m"
        memory: "512Mi"
```

**Enforcement:**

```yaml
# User tries to create pod exceeding max:
resources:
  requests:
    cpu: "3"  # Exceeds max of 2

# Rejected: "cpu limit exceeds maximum"
```

---



## Production Best Practices 


**1. Always Set Resource Requests and Limits:**

```yaml
resources:
  requests:
    memory: "256Mi"
    cpu: "250m"
  limits:
    memory: "512Mi"
    cpu: "500m"
```

Without requests: Scheduler can't make intelligent decisions.
Without limits: One pod can starve others.

**2. Health Checks Are Critical:**

```yaml
livenessProbe:
  httpGet:
    path: /health
    port: 8000
  initialDelaySeconds: 30
  periodSeconds: 10

readinessProbe:
  httpGet:
    path: /ready
    port: 8000
  initialDelaySeconds: 5
  periodSeconds: 5
```

- **Liveness:** Restart if application crashed/hung
- **Readiness:** Remove from service if not ready for traffic

**3. Use Labels Strategically:**

```yaml
metadata:
  labels:
    app: myapp
    version: v1.2.3
    environment: production
    team: backend
    cost-center: engineering
```

Enables powerful queries:
```bash
kubectl get pods -l environment=production,team=backend
```

**4. Implement PodDisruptionBudgets:**

```yaml
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: myapp-pdb
spec:
  minAvailable: 2
  # Always keep at least 2 pods running
  selector:
    matchLabels:
      app: myapp
```

Prevents cluster upgrades/maintenance from killing all pods at once.

**5. Security - RBAC:**

```yaml
# ServiceAccount for application
apiVersion: v1
kind: ServiceAccount
metadata:
  name: myapp-sa
---
# Role (what permissions)
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: pod-reader
rules:
- apiGroups: [""]
  resources: ["pods"]
  verbs: ["get", "list"]
---
# RoleBinding (who has permissions)
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: read-pods
subjects:
- kind: ServiceAccount
  name: myapp-sa
roleRef:
  kind: Role
  name: pod-reader
  apiGroup: rbac.authorization.k8s.io
```

Principle of least privilege: Only grant necessary permissions.

**6. Network Policies for Isolation:**

```yaml
# Default deny all
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny
spec:
  podSelector: {}
  policyTypes:
  - Ingress
  - Egress

# Then allow specific traffic
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-frontend-to-api
spec:
  podSelector:
    matchLabels:
      app: api
  ingress:
  - from:
    - podSelector:
        matchLabels:
          app: frontend
```

**7. Use Namespaces for Organization:**

```
production/      # Production workloads
├── api
├── database
└── cache

staging/         # Staging environment
├── api
└── database

development/     # Dev environment

monitoring/      # Monitoring tools (Prometheus, Grafana)

kube-system/     # System components (don't touch!)
```

**8. ConfigMaps and Secrets Management:**

```yaml
# Store config in ConfigMap
apiVersion: v1
kind: ConfigMap
metadata:
  name: app-config
data:
  log_level: INFO

# Secrets in Secret (external provider better)
apiVersion: v1
kind: Secret
metadata:
  name: app-secrets
type: Opaque
data:
  api_key: <base64-encoded>

# Or use external secrets
# - HashiCorp Vault
# - AWS Secrets Manager
# - Google Secret Manager
# - Azure Key Vault
```

**9. Pod Anti-Affinity for High Availability:**

```yaml
spec:
  affinity:
    podAntiAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
      - labelSelector:
          matchExpressions:
          - key: app
            operator: In
            values:
            - myapp
        topologyKey: kubernetes.io/hostname
# Don't schedule two pods of same app on same node
```

Ensures pods spread across nodes. Node failure doesn't kill all replicas.

**10. Rolling Updates with Care:**

```yaml
spec:
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
# maxUnavailable: 0 = zero downtime
```

**11. Monitoring and Logging:**

Deploy monitoring stack:

```bash
# Prometheus for metrics
kubectl apply -f prometheus.yaml

# Grafana for visualization
kubectl apply -f grafana.yaml

# Loki or ELK for logs
kubectl apply -f loki.yaml
```

Essential metrics:
- Pod CPU/memory usage
- Request latency (p50, p95, p99)
- Error rate
- Pod restart count
- Node resource utilization

**12. GitOps Workflow:**

```
1. All Kubernetes manifests in Git
2. Changes via Pull Requests
3. Code review + approval
4. Auto-deploy to cluster (ArgoCD/Flux)
5. Audit trail in Git history
```

Never `kubectl apply` manually in production!

### Data Engineering Specific

**1. Spark on Kubernetes:**

```yaml
apiVersion: sparkoperator.k8s.io/v1beta2
kind: SparkApplication
metadata:
  name: spark-etl
spec:
  type: Python
  mode: cluster
  image: "spark:3.5.0-python3"
  mainApplicationFile: "s3a://bucket/etl.py"
  
  driver:
    cores: 1
    coreLimit: "1200m"
    memory: "2g"
    serviceAccount: spark
  
  executor:
    cores: 2
    instances: 10
    memory: "4g"
  
  deps:
    packages:
    - org.apache.hadoop:hadoop-aws:3.3.4
```

**2. Airflow on Kubernetes (KubernetesExecutor):**

```yaml
# airflow.cfg
[core]
executor = KubernetesExecutor

[kubernetes]
namespace = airflow
worker_container_repository = apache/airflow
worker_container_tag = 2.7.3
delete_worker_pods = True
delete_worker_pods_on_failure = False
```

Each task runs in separate pod. Dynamic scaling, resource isolation.

**3. Job Patterns for ETL:**

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: daily-etl
spec:
  backoffLimit: 3
  activeDeadlineSeconds: 3600
  template:
    spec:
      restartPolicy: OnFailure
      containers:
      - name: etl
        image: etl:v1
        resources:
          requests:
            memory: "8Gi"
            cpu: "4"
          limits:
            memory: "16Gi"
            cpu: "8"
        volumeMounts:
        - name: data
          mountPath: /data
      volumes:
      - name: data
        persistentVolumeClaim:
          claimName: etl-data
```

**4. Handling Large Datasets:**

```yaml
# Use PVC for intermediate data
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: processing-data
spec:
  accessModes:
  - ReadWriteOnce
  resources:
    requests:
      storage: 500Gi
  storageClassName: fast-ssd

# Or object storage (S3, GCS)
env:
- name: AWS_ACCESS_KEY_ID
  valueFrom:
    secretKeyRef:
      name: aws-credentials
      key: access-key-id
- name: INPUT_PATH
  value: s3://data-lake/raw/
- name: OUTPUT_PATH
  value: s3://data-lake/processed/
```

### Troubleshooting Guide

**Pod Not Starting:**

```bash
# Check pod status
kubectl get pod myapp-abc123
# STATUS: ImagePullBackOff

# Describe for details
kubectl describe pod myapp-abc123
# Shows: Failed to pull image "myapp:v99": not found

# Fix: Correct image name/tag
```

**CrashLoopBackOff:**

```bash
kubectl logs myapp-abc123
# Shows application error (e.g., can't connect to database)

# Check previous logs (if container restarted)
kubectl logs myapp-abc123 --previous

# Check liveness probe settings (too aggressive?)
kubectl describe pod myapp-abc123
# Liveness: http-get http://:8000/health delay=5s
```

**Pod Pending:**

```bash
kubectl describe pod myapp-abc123
# Events:
#   0/3 nodes are available: insufficient cpu

# Options:
# 1. Add more nodes (Cluster Autoscaler)
# 2. Reduce resource requests
# 3. Scale down other workloads
```

**Service Not Reachable:**

```bash
# Check service
kubectl get svc myapp-service
# CLUSTER-IP: 10.96.0.10

# Check endpoints (should list pod IPs)
kubectl get endpoints myapp-service
# If empty: selector doesn't match pods

# Check pod labels
kubectl get pods --show-labels
# Ensure labels match service selector

# Test from another pod
kubectl run -it --rm debug --image=alpine --restart=Never -- sh
/ # wget -O- http://myapp-service
```

**DNS Issues:**

```bash
# Test DNS
kubectl run -it --rm debug --image=alpine --restart=Never -- sh
/ # nslookup myapp-service
# If fails: CoreDNS issue

# Check CoreDNS
kubectl get pods -n kube-system -l k8s-app=kube-dns
```

**High Memory Usage:**

```bash
kubectl top pods
# Shows current usage

# Check if hitting limits
kubectl describe pod myapp-abc123
# State: Running
# Last State: Terminated (OOMKilled)

# Solution: Increase memory limits
```

---



## Summary

This comprehensive guide covered:

**Docker:**
- Containerization fundamentals and why containers solve "works on my machine"
- Image layers, copy-on-write, and optimization strategies
- Networking: bridge, host, overlay networks, DNS resolution
- Volumes: persistent storage patterns and lifecycle
- Python-specific optimizations and multi-stage builds
- Docker Compose for multi-container applications

**Kubernetes:**
- Architecture: control plane, worker nodes, component responsibilities
- Pods: fundamental unit, lifecycle, multi-container patterns
- Workloads: Deployments (stateless), StatefulSets (stateful), DaemonSets, Jobs
- Networking: Service abstraction, DNS, Ingress for HTTP routing, Network Policies
- Storage: PV/PVC abstraction, StorageClass dynamic provisioning
- Autoscaling: HPA (horizontal), VPA (vertical), Cluster Autoscaler
- Resource management: requests/limits, QoS classes, quotas
- Production patterns: health checks, rolling updates, security (RBAC, Network Policies)

**Key Takeaways:**

1. **Start Simple:** Local development with Docker, deploy to Kubernetes when you need orchestration
2. **Resource Limits Always:** Set requests and limits for predictable scheduling and fair resource sharing
3. **Health Checks Critical:** Liveness and readiness probes prevent downtime
4. **Monitoring Essential:** Deploy Prometheus/Grafana, watch your metrics
5. **Security First:** Non-root containers, RBAC, Network Policies, secret management
6. **GitOps Workflow:** All configs in Git, automated deployments, audit trail

**Next Steps:**

1. **Hands-On:** Set up Minikube/kind, deploy sample applications
2. **Learn More:** Official Kubernetes documentation (kubernetes.io/docs)
3. **Certifications:** CKA (Administrator), CKAD (Developer), CKS (Security)
4. **Advanced Topics:** Service Mesh (Istio/Linkerd), Helm charts, Operators
5. **Production:** Learn your cloud provider's Kubernetes (EKS/GKE/AKS)